#### ML Part

In [10]:
import time
import string
import torch
from typing import List, Tuple, Optional, Dict
from itertools import product
from datetime import datetime
import os


import sys
sys.path.append(r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\Decoders\Hybrid")
# This part is just importing Travis's model but things I learned
# Has to be in .py, CAN't be in .pynb format otherwise it can't import it
from enigma_ml_model import EnigmaRotorClassifier, EnigmaDataset, predict_positions


class EnigmaMachine:
    """3-rotor Enigma machine implementation"""
    
    ROTORS = {
        'I':   'EKMFLGDQVZNTOWYHXUSPAIBRCJ',
        'II':  'AJDKSIRUXBLHWTMCQGZNPYFVOE',
        'III': 'BDFHJLCPRTXVZNYEIWGAKMUSQO',
        'IV':  'ESOVPZJAYQUIRHXLNFTGKDCMWB',
        'V':   'VZBRGITYUPSDNHLXAWMJQOFECK'
    }
    
    NOTCHES = {
        'I': 'Q', 'II': 'E', 'III': 'V', 'IV': 'J', 'V': 'Z'
    }
    
    REFLECTOR_B = 'YRUHQSLDPXNGOKMIEBFZCWVJAT'
    ALPHABET = string.ascii_uppercase
    
    def __init__(self, rotors: Tuple[str, str, str], positions: Tuple[int, int, int],
                 ring_settings: Tuple[int, int, int] = (0, 0, 0),
                 plugboard: Dict[str, str] = None):
        self.rotors = [self.ROTORS[r] for r in rotors]
        self.rotor_names = rotors
        self.positions = list(positions)
        self.ring_settings = list(ring_settings)
        self.notches = [self.NOTCHES[r] for r in rotors]
        self.reflector = self.REFLECTOR_B
        self.plugboard = plugboard or {}
    
    def _plugboard_swap(self, char: str) -> str:
        return self.plugboard.get(char, char)
    
    def _rotate_rotors(self):
        if self.ALPHABET[self.positions[1]] == self.notches[1]:
            self.positions[1] = (self.positions[1] + 1) % 26
            self.positions[2] = (self.positions[2] + 1) % 26
        elif self.ALPHABET[self.positions[0]] == self.notches[0]:
            self.positions[1] = (self.positions[1] + 1) % 26
        self.positions[0] = (self.positions[0] + 1) % 26
    
    def _encode_through_rotor(self, char_index: int, rotor_index: int, forward: bool = True) -> int:
        rotor = self.rotors[rotor_index]
        position = self.positions[rotor_index]
        ring = self.ring_settings[rotor_index]
        
        if forward:
            shifted = (char_index + position - ring) % 26
            encoded = self.ALPHABET.index(rotor[shifted])
            return (encoded - position + ring) % 26
        else:
            shifted = (char_index + position - ring) % 26
            encoded = rotor.index(self.ALPHABET[shifted])
            return (encoded - position + ring) % 26
    
    def _encode_through_reflector(self, char_index: int) -> int:
        return self.ALPHABET.index(self.reflector[char_index])
    
    def encrypt_char(self, char: str) -> str:
        if char not in self.ALPHABET:
            return char
        
        self._rotate_rotors()
        char = self._plugboard_swap(char)
        char_index = self.ALPHABET.index(char)
        
        for i in range(3):
            char_index = self._encode_through_rotor(char_index, i, forward=True)
        
        char_index = self._encode_through_reflector(char_index)
        
        for i in range(2, -1, -1):
            char_index = self._encode_through_rotor(char_index, i, forward=False)
        
        result = self.ALPHABET[char_index]
        result = self._plugboard_swap(result)
        return result
    
    def encrypt(self, text: str) -> str:
        return ''.join(self.encrypt_char(c.upper()) for c in text if c.isalpha())

#### Bombe Part

In [11]:
class BombeMultiMatch:
    """Modified Bombe that returns ALL possible matches for weak cribs"""
    
    def __init__(self, rotors_to_test: List[Tuple[str, str, str]] = None):
        if rotors_to_test is None:
            self.rotors_to_test = [
                ('I', 'II', 'III'),
                ('I', 'III', 'II'),
                ('II', 'I', 'III'),
                ('II', 'III', 'I'),
                ('III', 'I', 'II'),
                ('III', 'II', 'I')
            ]
        else:
            self.rotors_to_test = rotors_to_test
    
    def _test_crib(self, ciphertext: str, crib: str, rotors: Tuple[str, str, str],
                   rotor_positions: Tuple[int, int, int]) -> bool:
        try:
            enigma = EnigmaMachine(rotors, rotor_positions)
            encrypted_crib = enigma.encrypt(crib)
            return encrypted_crib == ciphertext[:len(crib)]
        except:
            return False
    
    def find_all_matches(self, ciphertext: str, crib: str, 
                        max_matches: int = 50,
                        verbose: bool = True) -> List[Dict]:
        """
        Find ALL settings that match the crib (not just the first one)
        
        Returns:
            List of matching settings with their details
        """
        crib = crib.upper().replace(' ', '')
        ciphertext = ciphertext.upper().replace(' ', '')
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"BOMBE: Finding ALL matches for crib '{crib}'")
            print(f"{'='*70}\n")
        
        matches = []
        start_time = time.time()
        tests_performed = 0
        
        for rotor_config in self.rotors_to_test:
            if verbose:
                print(f"Testing rotors: {rotor_config}")
            
            for pos in product(range(26), range(26), range(26)):
                tests_performed += 1
                
                if self._test_crib(ciphertext, crib, rotor_config, pos):
                    matches.append({
                        'rotors': rotor_config,
                        'positions': pos,
                        'positions_letters': f"{chr(65+pos[0])}{chr(65+pos[1])}{chr(65+pos[2])}"
                    })
                    
                    if verbose:
                        print(f"  ✓ Match #{len(matches)}: {rotor_config} @ {pos}")
                    
                    if len(matches) >= max_matches:
                        break
            
            if len(matches) >= max_matches:
                break
        
        elapsed = time.time() - start_time
        
        if verbose:
            print(f"\n{'='*70}")
            print(f"Bombe found {len(matches)} possible matches")
            print(f"Tests performed: {tests_performed:,}")
            print(f"Time: {elapsed:.2f} seconds")
            print(f"{'='*70}\n")
        
        return matches

#### Combining Part

In [12]:
class HybridCryptanalysis:
    """
    Combines Bombe and ML Model for optimal cryptanalysis
    
    Strategy:
    1. Bombe with weak crib → narrows to ~20 candidates
    2. ML Model ranks those 20 → picks top 5
    3. Bombe verifies top 5 → 100% accurate answer
    """
    
    def __init__(self, ml_model_path: str, device='cuda'):
        """
        Args:
            ml_model_path: Path to trained ML model (.pth file)
            device: 'cuda' or 'cpu'
        """
        self.bombe = BombeMultiMatch()
        self.device = device
        
        # Load ML model
        print(f"Loading ML model from {ml_model_path}...")
        self.ml_model = EnigmaRotorClassifier(message_length=100, hidden_dim=256)
        self.ml_model.load_state_dict(torch.load(ml_model_path, map_location=device))
        self.ml_model.to(device)
        self.ml_model.eval()
        print("✓ ML model loaded\n")
    
    def hybrid_attack(self, ciphertext: str, weak_crib: str, 
                     verification_crib: str = None,
                     verbose: bool = True) -> Dict:
        """
        Hybrid attack combining Bombe and ML
        
        Args:
            ciphertext: Encrypted text
            weak_crib: Short crib (1-2 words) to narrow search
            verification_crib: Longer crib to verify (optional)
            verbose: Print progress
        
        Returns:
            Dictionary with found settings and metrics
        """
        start_time = time.time()
        
        # STEP 1: Bombe narrows search space
        if verbose:
            print("\n" + "="*80)
            print("STEP 1: BOMBE NARROWING SEARCH SPACE")
            print("="*80)
        
        candidates = self.bombe.find_all_matches(
            ciphertext=ciphertext,
            crib=weak_crib,
            max_matches=50,
            verbose=verbose
        )
        
        if not candidates:
            return {
                'success': False,
                'message': 'Bombe found no matches with provided crib'
            }
        
        bombe_time = time.time() - start_time
        
        # STEP 2: ML Model ranks candidates
        if verbose:
            print("\n" + "="*80)
            print("STEP 2: ML MODEL RANKING CANDIDATES")
            print("="*80)
        
        ml_start = time.time()
        ranked_candidates = self._rank_candidates_with_ml(ciphertext, candidates, verbose)
        ml_time = time.time() - ml_start
        
        # STEP 3: Verify top candidates
        if verbose:
            print("\n" + "="*80)
            print("STEP 3: VERIFYING TOP CANDIDATES")
            print("="*80)
        
        verify_start = time.time()
        
        # Use verification crib if provided, otherwise use weak crib
        verify_crib = verification_crib if verification_crib else weak_crib
        
        for i, candidate in enumerate(ranked_candidates[:10], 1):  # Check top 10
            if verbose:
                print(f"\nTesting candidate #{i}:")
                print(f"  Rotors: {candidate['rotors']}")
                print(f"  Positions: {candidate['positions_letters']}")
                print(f"  ML Confidence: {candidate['ml_confidence']:.1f}%")
            
            # Full verification
            enigma = EnigmaMachine(candidate['rotors'], candidate['positions'])
            decrypted = enigma.encrypt(ciphertext)
            
            # Check if decryption makes sense (can use longer crib here)
            if verification_crib:
                expected = verification_crib.upper().replace(' ', '')
                if decrypted.startswith(expected):
                    verify_time = time.time() - verify_start
                    total_time = time.time() - start_time
                    
                    if verbose:
                        print(f"\n{'='*80}")
                        print("✓ SOLUTION VERIFIED!")
                        print(f"{'='*80}\n")
                    
                    return {
                        'success': True,
                        'rotors': candidate['rotors'],
                        'positions': candidate['positions'],
                        'positions_letters': candidate['positions_letters'],
                        'ml_confidence': candidate['ml_confidence'],
                        'bombe_candidates': len(candidates),
                        'verified_rank': i,
                        'timing': {
                            'bombe_time': bombe_time,
                            'ml_time': ml_time,
                            'verify_time': verify_time,
                            'total_time': total_time
                        }
                    }
        
        # If no verification, return top ML pick
        return {
            'success': True,
            'rotors': ranked_candidates[0]['rotors'],
            'positions': ranked_candidates[0]['positions'],
            'positions_letters': ranked_candidates[0]['positions_letters'],
            'ml_confidence': ranked_candidates[0]['ml_confidence'],
            'bombe_candidates': len(candidates),
            'verified': False,
            'timing': {
                'bombe_time': bombe_time,
                'ml_time': ml_time,
                'total_time': time.time() - start_time
            }
        }
    
    def _rank_candidates_with_ml(self, ciphertext: str, 
                                 candidates: List[Dict],
                                 verbose: bool = True) -> List[Dict]:
        """
        Use ML model to rank candidate settings by confidence
        """
        if verbose:
            print(f"\nRanking {len(candidates)} candidates with ML model...")
        
        # Get ML prediction for the ciphertext
        ml_predictions, ml_confidences = predict_positions(
            self.ml_model, 
            ciphertext[:100],  # Use first 100 chars
            device=self.device
        )
        
        if verbose:
            print(f"ML predicts positions: {ml_predictions} ({chr(ml_predictions[0]+65)}{chr(ml_predictions[1]+65)}{chr(ml_predictions[2]+65)})")
            print(f"ML confidences: R1={ml_confidences[0]:.1f}%, R2={ml_confidences[1]:.1f}%, R3={ml_confidences[2]:.1f}%")
        
        # Score each candidate based on how close it is to ML prediction
        for candidate in candidates:
            pos = candidate['positions']
            
            # Calculate similarity score
            matches = sum(1 for i in range(3) if pos[i] == ml_predictions[i])
            
            # Weighted confidence based on matches
            if matches == 3:
                confidence = 95.0  # Perfect match
            elif matches == 2:
                confidence = 70.0  # 2 out of 3
            elif matches == 1:
                confidence = 40.0  # 1 out of 3
            else:
                confidence = 10.0  # No matches
            
            candidate['ml_confidence'] = confidence
            candidate['ml_matches'] = matches
        
        # Sort by confidence (highest first)
        ranked = sorted(candidates, key=lambda x: x['ml_confidence'], reverse=True)
        
        if verbose:
            print(f"\nTop 5 candidates by ML confidence:")
            for i, c in enumerate(ranked[:5], 1):
                print(f"  {i}. {c['rotors']} @ {c['positions_letters']} - Confidence: {c['ml_confidence']:.1f}%")
        
        return ranked

#### Main part to run

Some things to kind of think about
* bombe relies on crib but where that crib is in the text doesn't matter
* a good crib is one that makes lots of contradictions = eliminates a bunch of possible rotor configs
* weak crib allows a lot of possibilites
* weak crib = short length, repeats part of its own letters ex) according where it has two c's
* 

In [16]:
def main():
    print("\n" + "="*80)
    print("HYBRID CRYPTANALYSIS SYSTEM")
    print("Bombe + Machine Learning")
    print("="*80 + "\n")
    
    # Configuration
    encrypted_file = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Discussion1.txt"
    ml_model_path = "best_enigma_model.pth"
    
    # Read encrypted file
    print(f"Reading encrypted file: {encrypted_file}")
    with open(encrypted_file, 'r', encoding='utf-8') as f:
        ciphertext = f.read().strip()
    
    ciphertext = ''.join(c for c in ciphertext if c.isalpha()).upper()
    print(f"Ciphertext length: {len(ciphertext)} characters\n")
    
    # Initialize hybrid system
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    hybrid = HybridCryptanalysis(ml_model_path, device=device)
    
    # Run hybrid attack
    result = hybrid.hybrid_attack(
        ciphertext=ciphertext,
        weak_crib="DISCUSSION",  # Short crib (6 letters or less)
        verification_crib="DISCUSSIONTOPICICHOSE",  # Longer verification
        verbose=True
    )
    
    # Display results
    if result['success']:
        print("\n" + "="*80)
        print("HYBRID ATTACK SUCCESS!")
        print("="*80)
        print(f"\nFound Settings:")
        print(f"  Rotors: {result['rotors']}")
        print(f"  Positions: {result['positions_letters']} (numeric: {result['positions']})")
        print(f"  ML Confidence: {result['ml_confidence']:.1f}%")
        print(f"\nPerformance:")
        print(f"  Bombe found {result['bombe_candidates']} candidates")
        print(f"  Verified at rank: {result.get('verified_rank', 'N/A')}")
        print(f"\nTiming:")
        print(f"  Bombe time: {result['timing']['bombe_time']:.2f}s")
        print(f"  ML ranking time: {result['timing']['ml_time']:.2f}s")
        if 'verify_time' in result['timing']:
            print(f"  Verification time: {result['timing']['verify_time']:.2f}s")
        print(f"  Total time: {result['timing']['total_time']:.2f}s")
        print("="*80 + "\n")
    else:
        print("\n✗ Hybrid attack failed")
        print(result['message'])


if __name__ == "__main__":
    main()


HYBRID CRYPTANALYSIS SYSTEM
Bombe + Machine Learning

Reading encrypted file: C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Discussion1.txt
Ciphertext length: 2762 characters

Loading ML model from best_enigma_model.pth...
✓ ML model loaded


STEP 1: BOMBE NARROWING SEARCH SPACE

BOMBE: Finding ALL matches for crib 'DISCUSSION'

Testing rotors: ('I', 'II', 'III')
Testing rotors: ('I', 'III', 'II')
Testing rotors: ('II', 'I', 'III')
Testing rotors: ('II', 'III', 'I')
Testing rotors: ('III', 'I', 'II')
Testing rotors: ('III', 'II', 'I')

Bombe found 0 possible matches
Tests performed: 105,456
Time: 8.52 seconds


✗ Hybrid attack failed
Bombe found no matches with provided crib
